# 99 — Scratch: ACT (Average Concentration Time) Verification

This notebook verifies the `act` parameter added to `CN3SParams`, the `pd.Series` path in `CN3S.run()`, and the `CN3SOptimizer` integration.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import pandas as pd

from cn3s import CN3S, CN3SOptimizer, CN3SParams

## 1 — CN3SParams: round-trip test

In [ ]:
# as_list() should now return 8 elements
p = CN3SParams(area=9642.0, act=4)
vec = p.as_list()
print(f"as_list length: {len(vec)}  (expected 8)")
print(f"vec[-1] = {vec[-1]}  (expected 4.0)")
assert len(vec) == 8
assert vec[-1] == 4.0

# from_vector should round act to int
p2 = CN3SParams.from_vector(vec, area=9642.0)
print(f"p2.act = {p2.act!r}  (expected 4, type int)")
assert p2.act == 4
assert isinstance(p2.act, int)

# Float rounding: 4.6 → 5
vec_round = list(vec)
vec_round[-1] = 4.6
p3 = CN3SParams.from_vector(vec_round, area=9642.0)
print(f"p3.act (from 4.6) = {p3.act!r}  (expected 5)")
assert p3.act == 5

print("\nAll CN3SParams checks PASSED")

## 2 — CN3S.run() with a pd.Series: index is set automatically

In [ ]:
import numpy as np

ACT = 4
N = 100

# Build a synthetic dated precipitation Series
dates = pd.date_range("2010-01-01", periods=N, freq="D")
prec_series = pd.Series(np.random.default_rng(42).uniform(0, 10, size=N), index=dates, name="prec")

params = CN3SParams(area=9642.0, act=ACT)
model = CN3S(params)

model.run(prec_series)  # pd.Series path

expected_first_date = dates[3] + pd.Timedelta(days=ACT)
actual_first_date = model.results.index[0]

print(f"Expected results.index[0]: {expected_first_date.date()}")
print(f"Actual  results.index[0]: {actual_first_date.date()}")
assert actual_first_date == expected_first_date, "Index mismatch!"
print(f"results shape: {model.results.shape}  (expected ({N - 3}, 11))")
assert model.results.shape == (N - 3, 11)

model.results.head()

## 3 — CN3S.run() with a plain list: RangeIndex preserved

In [ ]:
model_list = CN3S(CN3SParams(area=9642.0, act=ACT))
model_list.run(prec_series.tolist())  # list path — no index management

print(f"results.index type: {type(model_list.results.index).__name__}  (expected RangeIndex)")
assert isinstance(model_list.results.index, pd.RangeIndex), "Expected RangeIndex for list input"
print("List-path check PASSED")

## 4 — Load real data (requires /data/CN3S mount)

In [ ]:
BASIN = "BarraDosBugres"
STATION = 66010000
BASE_FOLDER = "/data/CN3S/basins/"

path = Path(BASE_FOLDER) / BASIN
assert path.exists(), f"Path {path} does not exist — skipping real-data cells"

prec = pd.read_parquet(path / f"Daily_Prec_{STATION}.parquet")
prec.index = prec.index - pd.Timedelta(hours=12)  # mandatory 12h correction

q = pd.read_parquet(path / f"Daily_Q_{STATION}.parquet")

print(f"prec: {len(prec)} rows, {prec.index[0].date()} → {prec.index[-1].date()}")
print(f"q   : {len(q)} rows, {q.index[0].date()} → {q.index[-1].date()}")

## 5 — CN3S.run() with real dated series (act=4)

In [ ]:
AREA = 9642  # km²
ACT = 4

model_real = CN3S(CN3SParams(area=AREA, act=ACT))
model_real.run(prec["prec"])  # pd.Series — index auto-assigned with ACT shift

print(f"results.index[0] : {model_real.results.index[0].date()}")
print(f"prec.index[3]+4d : {(prec.index[3] + pd.Timedelta(days=ACT)).date()}")
assert model_real.results.index[0] == prec.index[3] + pd.Timedelta(days=ACT)

model_real.results.head()

## 6 — CN3SOptimizer: verify act is optimized (short run)

In [ ]:
sliced_prec = prec.iloc[-1000:-500]

model_opt = CN3S(CN3SParams(area=AREA))
optim = CN3SOptimizer(
    prec=sliced_prec["prec"],
    observed_q=q["Vazao"],
    model=model_opt,
    train_ratio=0.7,
    max_act=10,
)

print(f"Bounds count: {len(optim._bounds_list)}  (expected 8)")
print(f"act bound: {optim._bounds_list[-1]}  (expected (0.0, 10.0))")
assert len(optim._bounds_list) == 8
assert optim._bounds_list[-1] == (0.0, 10.0)

print(f"Train cutoff date: {optim._train_cutoff.date()}")
print(f"Aligned record: {len(optim.aligned)} rows")

In [ ]:
# Short optimization run — just 5 iterations to verify end-to-end flow
optim.optimize(maxiter=5, tol=1e-2, disp=False, workers=1)

In [ ]:
print(f"Optimized params: {optim.model.params}")
print(f"act type: {type(optim.model.params.act).__name__}  (expected int)")
assert isinstance(optim.model.params.act, int)
print(f"results index type: {type(optim.model.results.index).__name__}")
print(f"results.index[0]: {optim.model.results.index[0].date()}")
print("\nOptimizer ACT integration PASSED")

In [ ]:
optim.model.results.head()